# Concealed Weapon Detection (CWD)

This notebook develops and evaluates the PMMW concealed-weapon object-detection component of the multi-modal threat prediction system.

## Planned workflow

1. Audit and clean the PMMW dataset
2. Visualise images and bounding-box annotations
3. Create leakage-safe training, validation and test splits
4. Convert annotations to the detector's required format
5. Train a lightweight object-detection baseline
6. Evaluate detection accuracy, errors and edge-deployment cost
7. Compare controlled improvements and select the final CWD model

## 1. Dataset setup and parsing

The PMMW dataset uses Pascal VOC XML annotations. The following cells parse every image, object label and bounding box into simple pandas DataFrames.

In [ ]:
from pathlib import Path
import hashlib
import filecmp
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

SEED = 42
DATASET_ROOT = Path("sensors-722939-PMMW-data")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path("CWD") / DATASET_ROOT

IMAGE_DIR = DATASET_ROOT / "images"
ANNOTATION_DIR = DATASET_ROOT / "Annotations"

if not IMAGE_DIR.exists() or not ANNOTATION_DIR.exists():
    raise FileNotFoundError(f"PMMW dataset not found at {DATASET_ROOT.resolve()}")

In [ ]:
image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
xml_paths = sorted(ANNOTATION_DIR.glob("*.xml"))
records = []

for xml_path in xml_paths:
    root = ET.parse(xml_path).getroot()
    width = int(root.findtext("size/width"))
    height = int(root.findtext("size/height"))

    for obj in root.findall("object"):
        box = obj.find("bndbox")
        xmin, ymin, xmax, ymax = [
            int(box.findtext(name)) for name in ("xmin", "ymin", "xmax", "ymax")
        ]
        records.append({
            "image_name": f"{xml_path.stem}.jpg",
            "class_name": obj.findtext("name").strip().lower(),
            "width": width, "height": height,
            "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
        })

annotations = pd.DataFrame(records)
annotations["box_width"] = annotations["xmax"] - annotations["xmin"]
annotations["box_height"] = annotations["ymax"] - annotations["ymin"]
annotations["relative_box_area"] = (
    annotations["box_width"] * annotations["box_height"]
    / (annotations["width"] * annotations["height"])
)
annotations["invalid_box"] = (
    (annotations["xmin"] < 0) | (annotations["ymin"] < 0)
    | (annotations["xmax"] > annotations["width"])
    | (annotations["ymax"] > annotations["height"])
    | (annotations["box_width"] <= 0) | (annotations["box_height"] <= 0)
)

per_image = annotations.groupby("image_name").agg(
    object_count=("class_name", "size"),
    classes=("class_name", lambda values: ", ".join(sorted(set(values)))),
    has_gun=("class_name", lambda values: "gun" in set(values)),
).reset_index()

images = pd.DataFrame({
    "image_name": [path.name for path in image_paths],
    "image_path": image_paths,
}).merge(per_image, on="image_name", how="left")
images["has_annotation"] = images["object_count"].notna()
images["object_count"] = images["object_count"].fillna(0).astype(int)
images["has_gun"] = images["has_gun"].fillna(False).astype(bool)

## 2. Dataset statistics

In [ ]:
image_names = {path.stem for path in image_paths}
xml_names = {path.stem for path in xml_paths}
missing_xml = sorted(image_names - xml_names)
missing_images = sorted(xml_names - image_names)
annotated_images = images[images["has_annotation"]]

print(f"Image files:             {len(image_paths):,}")
print(f"XML annotation files:    {len(xml_paths):,}")
print(f"Total bounding boxes:    {len(annotations):,}")
print(f"Gun-positive images:     {annotated_images['has_gun'].sum():,}")
print(f"Gun-negative images:     {(~annotated_images['has_gun']).sum():,}")
print(f"Images without XML:      {len(missing_xml):,}")
print(f"XML files without image: {len(missing_images):,}")
print(f"Invalid bounding boxes:  {annotations['invalid_box'].sum():,}")
print(f"Image dimensions:        {sorted(set(zip(annotations.width, annotations.height)))}")

if missing_xml:
    print("\nImages without annotations:", ", ".join(missing_xml))

In [ ]:
class_counts = annotations["class_name"].value_counts().sort_values(ascending=False)
gun_status = pd.Series({
    "Gun present": int(annotated_images["has_gun"].sum()),
    "No gun": int((~annotated_images["has_gun"]).sum()),
})
boxes_per_image = annotated_images["object_count"].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_counts.plot.bar(ax=axes[0], color="steelblue", title="Objects by class")
gun_status.plot.bar(ax=axes[1], color=["firebrick", "slategray"], title="Annotated images")
boxes_per_image.plot.bar(ax=axes[2], color="seagreen", title="Bounding boxes per image")
axes[0].set_ylabel("Count")
axes[1].set_ylabel("Images")
axes[2].set(xlabel="Number of boxes", ylabel="Images")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Annotation examples

In [ ]:
sample_names = annotated_images.sample(6, random_state=SEED)["image_name"]
colours = {"gun": "red", "person": "cyan", "other": "orange", "cat": "yellow"}
fig, axes = plt.subplots(2, 3, figsize=(10, 12))

for ax, image_name in zip(axes.flat, sample_names):
    ax.imshow(Image.open(IMAGE_DIR / image_name), cmap="gray")
    for _, box in annotations[annotations["image_name"] == image_name].iterrows():
        colour = colours.get(box["class_name"], "lime")
        ax.add_patch(Rectangle(
            (box["xmin"], box["ymin"]), box["box_width"], box["box_height"],
            fill=False, edgecolor=colour, linewidth=2,
        ))
        ax.text(box["xmin"], box["ymin"], box["class_name"], color=colour, fontsize=9)
    ax.set_title(image_name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Exact duplicate detection

Each image is hashed using SHA-256 over its complete file contents. Files sharing a hash are then compared byte-for-byte. This detects only exact copies; visually similar frames are not counted.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

images["sha256"] = images["image_path"].apply(sha256_file)
duplicate_groups = images.groupby("sha256")["image_path"].apply(list)
duplicate_groups = duplicate_groups[duplicate_groups.apply(len) > 1]

# Confirm every hash match by comparing the complete file contents.
for paths in duplicate_groups:
    if not all(filecmp.cmp(paths[0], path, shallow=False) for path in paths[1:]):
        raise RuntimeError("A hash group failed byte-for-byte verification.")

duplicate_files = sum(len(paths) for paths in duplicate_groups)
extra_copies = sum(len(paths) - 1 for paths in duplicate_groups)

print("Duplicate method: SHA-256 + byte-for-byte verification")
print(f"Exact duplicate groups:  {len(duplicate_groups):,}")
print(f"Files in those groups:   {duplicate_files:,}")
print(f"Extra duplicate copies:  {extra_copies:,}")
print(f"Unique image contents:   {images['sha256'].nunique():,}")
print("Verification:            all duplicate groups are byte-identical")

print("\nExample duplicate groups:")
for paths in duplicate_groups.head(5):
    print("  " + ", ".join(path.name for path in paths))

In [ ]:
group_sizes = duplicate_groups.apply(len).value_counts().sort_index()
ax = group_sizes.plot.bar(figsize=(7, 4), color="mediumpurple")
ax.set(title="Exact duplicate group sizes", xlabel="Files with identical contents", ylabel="Number of groups")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
examples = duplicate_groups.head(3).tolist()
if examples:
    fig, axes = plt.subplots(len(examples), 2, figsize=(6, 4 * len(examples)))
    if len(examples) == 1:
        axes = [axes]
    for row, paths in zip(axes, examples):
        for ax, path in zip(row, paths[:2]):
            ax.imshow(Image.open(path), cmap="gray")
            ax.set_title(path.name)
            ax.axis("off")
    fig.suptitle("Examples confirmed as byte-identical", y=1.01)
    plt.tight_layout()
    plt.show()